In [3]:
# Standard library imports
import os
import re
import pickle

# Third-party data manipulation and visualization imports
import pandas as pd
import seaborn as sns
from tqdm import tqdm

# Machine Learning and NLP libraries
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Custom Persian NLP tools (shekar)
from shekar.preprocessing import EmojiRemover, PunctuationRemover, NonPersianRemover
from shekar import Normalizer, WordTokenizer, Lemmatizer, SentenceTokenizer
# Enable pandas progress apply
tqdm.pandas()


In [4]:
aiDataset = pd.read_csv("persian_ai_generated_final.csv",encoding="utf-8-sig")
humanDataset = pd.read_csv("blogs.csv",encoding="utf-8-sig")

In [5]:
aiDataset

,topic,generated_text,model,provider,prompt_style,temperature,top_p,char_count,word_count
0,ارکستراسیون و کوبرنتیز,ارکستراسیون و کوبرنتیز دو اصطلاحی هستند که در ...,meta-llama/llama-3.1-8b-instruct,openrouter,qa_explainer,0.45,0.85,1451.0,262.0
1,چگونه می‌توان از استعدادهای جهانی برای پیشبرد ...,استفاده از استعدادهای جهانی برای پیشبرد فناوری...,meta-llama/llama-3.1-8b-instruct,openrouter,opinion_critical,0.54,0.89,2241.0,378.0
2,آیا می‌توان با استفاده از انرژی خورشیدی به سیا...,**مقدمه:**\nسفر به سیارات دیگر یکی از بزرگ‌تری...,openai/gpt-4o-mini,openrouter,summary_overview,0.30,0.92,1860.0,318.0
3,سیستم‌های پرداخت آنلاین و درگاه‌های پرداخت,حالا که در جهان این روزها همه چیز می‌تواند با ...,meta-llama/llama-3.1-8b-instruct,openrouter,conversational_blog,0.86,0.95,1101.0,209.0
4,سیستم‌های بهینه‌سازی مسیریابی در شبکه,سیستم‌های بهینه‌سازی مسیریابی در شبکه‌ها به من...,microsoft/phi-4,openrouter,qa_explainer,0.62,0.92,2685.0,444.0
...,...,...,...,...,...,...,...,...,...
10541,سیستم‌های مدیریت فرایندهای کسب‌وکار (BPMS),سیستم‌های مدیریت فرایندهای کسب‌وکار (BPMS یا B...,am/nemotron-3.5-lightning-30b-a3b,anymodel_endpoint,opinion_critical,0.57,0.98,4404.0,699.0
10542,طراحی داخلی بر اساس روانشناسی رنگ‌ها,**عنوان مقاله:**\n**بررسی تحلیلی و کاربردی روا...,am/nemotron-3-ultra-550b-a55b,anymodel_endpoint,formal_essay,0.34,0.88,10444.0,1612.0
10543,نانوتکنولوژی در پزشکی و صنعت,**عنوان مقاله: تحلیل جامع کاربردهای نانوتکنولو...,google/gemma-4-31b-it,NVIDIA,formal_essay,0.61,0.92,5033.0,807.0
10544,سیستم‌های پخش موسیقی و پادکست با پیشنهاددهی هو...,**نقد سیستم‌های پخش موسیقی و پادکست با پیشنهاد...,hy3,chat.b.ai,opinion_critical,0.41,0.94,3069.0,521.0


In [6]:
humanDataset

,text
0,شبکه‌های اجتماعی،هوموفیلی و اگزیستانسیالیم مسا...
1,شاد بودن از مد افتاده سلام چند روز پیش یهویی ا...
2,محراب حسین زهی - بازدید سردار شیخی از نمایشگاه...
3,فرهنگ سازمانی اگیلوی و رفقا 72 سال پیش ، دیوید...
4,جلوگیری از کودک آزاری یا سو استفاده جنسی در مد...
...,...
385895,همیشه برای جدا کردن فلش در ویندوز 10 از گزینه ...
385896,در هنگام خواندن یک مطلب هستید یا لیست خرید خود...
385897,اگر یک فایل ورد در اختیارتان قرار گرفته که به ...
385898,بسیاری از ما در هنگام تایپ با موبایل دچار اشتب...


In [7]:
aiDataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10546 entries, 0 to 10545
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   topic           10546 non-null  str    
 1   generated_text  10546 non-null  str    
 2   model           10546 non-null  str    
 3   provider        10546 non-null  str    
 4   prompt_style    10546 non-null  str    
 5   temperature     10546 non-null  float64
 6   top_p           10546 non-null  float64
 7   char_count      10546 non-null  float64
 8   word_count      10546 non-null  float64
dtypes: float64(4), str(5)
memory usage: 104.4 MB


In [8]:
humanDataset.isna().sum()

text    0
dtype: int64

In [9]:
humanDataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 385900 entries, 0 to 385899
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    385900 non-null  str  
dtypes: str(1)
memory usage: 2.1 GB


In [10]:
aiDataset

,topic,generated_text,model,provider,prompt_style,temperature,top_p,char_count,word_count
0,ارکستراسیون و کوبرنتیز,ارکستراسیون و کوبرنتیز دو اصطلاحی هستند که در ...,meta-llama/llama-3.1-8b-instruct,openrouter,qa_explainer,0.45,0.85,1451.0,262.0
1,چگونه می‌توان از استعدادهای جهانی برای پیشبرد ...,استفاده از استعدادهای جهانی برای پیشبرد فناوری...,meta-llama/llama-3.1-8b-instruct,openrouter,opinion_critical,0.54,0.89,2241.0,378.0
2,آیا می‌توان با استفاده از انرژی خورشیدی به سیا...,**مقدمه:**\nسفر به سیارات دیگر یکی از بزرگ‌تری...,openai/gpt-4o-mini,openrouter,summary_overview,0.30,0.92,1860.0,318.0
3,سیستم‌های پرداخت آنلاین و درگاه‌های پرداخت,حالا که در جهان این روزها همه چیز می‌تواند با ...,meta-llama/llama-3.1-8b-instruct,openrouter,conversational_blog,0.86,0.95,1101.0,209.0
4,سیستم‌های بهینه‌سازی مسیریابی در شبکه,سیستم‌های بهینه‌سازی مسیریابی در شبکه‌ها به من...,microsoft/phi-4,openrouter,qa_explainer,0.62,0.92,2685.0,444.0
...,...,...,...,...,...,...,...,...,...
10541,سیستم‌های مدیریت فرایندهای کسب‌وکار (BPMS),سیستم‌های مدیریت فرایندهای کسب‌وکار (BPMS یا B...,am/nemotron-3.5-lightning-30b-a3b,anymodel_endpoint,opinion_critical,0.57,0.98,4404.0,699.0
10542,طراحی داخلی بر اساس روانشناسی رنگ‌ها,**عنوان مقاله:**\n**بررسی تحلیلی و کاربردی روا...,am/nemotron-3-ultra-550b-a55b,anymodel_endpoint,formal_essay,0.34,0.88,10444.0,1612.0
10543,نانوتکنولوژی در پزشکی و صنعت,**عنوان مقاله: تحلیل جامع کاربردهای نانوتکنولو...,google/gemma-4-31b-it,NVIDIA,formal_essay,0.61,0.92,5033.0,807.0
10544,سیستم‌های پخش موسیقی و پادکست با پیشنهاددهی هو...,**نقد سیستم‌های پخش موسیقی و پادکست با پیشنهاد...,hy3,chat.b.ai,opinion_critical,0.41,0.94,3069.0,521.0


In [11]:
humanDataset = humanDataset.sample(10546,ignore_index=True)

In [12]:
datasetColumns=("text","aiGenerated")

In [13]:
aiTempDataSet = pd.DataFrame(columns=datasetColumns)
aiTempDataSet["text"] = aiDataset["generated_text"]
aiTempDataSet["aiGenerated"] = 1
aiTempDataSet

,text,aiGenerated
0,ارکستراسیون و کوبرنتیز دو اصطلاحی هستند که در ...,1
1,استفاده از استعدادهای جهانی برای پیشبرد فناوری...,1
2,**مقدمه:**\nسفر به سیارات دیگر یکی از بزرگ‌تری...,1
3,حالا که در جهان این روزها همه چیز می‌تواند با ...,1
4,سیستم‌های بهینه‌سازی مسیریابی در شبکه‌ها به من...,1
...,...,...
10541,سیستم‌های مدیریت فرایندهای کسب‌وکار (BPMS یا B...,1
10542,**عنوان مقاله:**\n**بررسی تحلیلی و کاربردی روا...,1
10543,**عنوان مقاله: تحلیل جامع کاربردهای نانوتکنولو...,1
10544,**نقد سیستم‌های پخش موسیقی و پادکست با پیشنهاد...,1


In [14]:
humanTempDataset =  pd.DataFrame(columns=datasetColumns)
humanTempDataset["text"] = humanDataset["text"]
humanTempDataset["aiGenerated"] = 0

In [15]:
humanTempDataset

,text,aiGenerated
0,ساختمان آلومینیوم ساختمانی تجاری در خیابان جمه...,0
1,(زاده 2 اردیبهشت 1338 - درگذشته 8 آبان 1386 ) ...,0
2,در جریان یک بیانیه از سوی ریحانا پرچت، نویسنده...,0
3,طرح اولیه‌ی یکی از تازه‌ترین پوسته‌هایی که به ...,0
4,"بازی ""داستان‌سرایی بازی جذابیت بالایی دارد و د...",0
...,...,...
10541,بدست آوردن شماره سیم کارت بدون تماس و اعتبار س...,0
10542,تفاوت بین Asynchronous و Multi - Threading پیا...,0
10543,تغییر در زندگی وقتی یه نصاب ساده دوربین مدار ب...,0
10544,چرا لاراول بهترین فریمورک PHP است؟ توسعه دهندگ...,0


In [16]:
def preprocess_persian_text(text: str):
    text = re.sub(r'\n'," " , text)
    pipeline = (
            EmojiRemover()
            | PunctuationRemover()
            | NonPersianRemover()
            | Normalizer()
            | Lemmatizer()
            | WordTokenizer()
        )
    tokens = pipeline(text)
    return ' '.join(tokens)


In [17]:
finalDataset = pd.concat([aiTempDataSet,humanTempDataset]).sample(frac=1,ignore_index=True)

In [18]:
finalDataset

,text,aiGenerated
0,احکام ماه مبارک رمضان احکام روزه 1 - روزه در ش...,0
1,اکنون شرکت آمادگی تولید چندین اسپیکر خانگی هوم...,0
2,سازمان پیش‌ازاین با انتشار یک توییت تازه به ای...,0
3,شش و نیم ماهگی رشد فرزند شماچپ دست یا راست دست...,0
4,با اینکه برای مدتی تبلت‌ها به بازار لپ‌تاپ‌ها ...,0
...,...,...
21087,نامه‌های رز، کتابی از کلاریس سابار روایت داستا...,0
21088,انگشت گذاری پیانو و تقویت انگشتان انگشت گذاری ...,0
21089,علی اطهری کرمانی (زاده مرداد 1305 - درگذشت 137...,0
21090,### بررسی کاربرد بلاک‌چین در افزایش شفافیت امو...,1


In [19]:
finalDataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 21092 entries, 0 to 21091
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text         21092 non-null  str  
 1   aiGenerated  21092 non-null  int64
dtypes: int64(1), str(1)
memory usage: 160.7 MB


In [20]:
finalDataset["text"].isna().sum()

np.int64(0)

In [21]:
# Standardized Dataset Loading and Saving
DATASET_PATH = "processedDataset.csv"

if not os.path.exists(DATASET_PATH):
    print(f"Saving processed dataset to {DATASET_PATH}...")
    # Save the dataset to a CSV file for future use
    finalDataset.to_csv(DATASET_PATH, encoding="utf-8-sig", index=False)
    print("Dataset saved successfully.")
else:
    print(f"Loading processed dataset from {DATASET_PATH}...")
    # Load the already processed dataset to save time
    finalDataset = pd.read_csv(DATASET_PATH)
    print("Dataset loaded successfully.")


Saving processed dataset to processedDataset.csv...
Dataset saved successfully.


In [22]:
# Better Loading and Saving using Joblib (Optimized for large numpy arrays/sklearn models)
from joblib import Parallel, delayed
import multiprocessing

VECTORIZER_PATH = "tfidf_vectorizer.joblib"
X_FEATURES_PATH = "tfidf_vectorizer_X.joblib"

if not os.path.exists(VECTORIZER_PATH) or not os.path.exists(X_FEATURES_PATH):
    print("Fitting TF-IDF Vectorizer on the dataset...")

    print("Preprocessing texts in parallel...")
    num_cores = multiprocessing.cpu_count()
    preprocessed_texts = Parallel(n_jobs=num_cores)(
        delayed(preprocess_persian_text)(text) for text in tqdm(finalDataset["text"])
    )

    # Initialize TF-IDF Vectorizer with n-grams (3,3)
    tfidf_vectorizer = TfidfVectorizer(
        ngram_range=(3, 3),
        strip_accents="unicode",
    )
    # Transform text data into TF-IDF features
    X = tfidf_vectorizer.fit_transform(preprocessed_texts)

    print("Saving Vectorizer and Features...")
    joblib.dump(tfidf_vectorizer, VECTORIZER_PATH)
    joblib.dump(X, X_FEATURES_PATH)
    print("Saved successfully.")
else:
    print("Loading TF-IDF Vectorizer and Features from disk...")
    # Load the pre-trained vectorizer and transformed features
    tfidf_vectorizer = joblib.load(VECTORIZER_PATH)
    X = joblib.load(X_FEATURES_PATH)
    print("Loaded successfully.")


Fitting TF-IDF Vectorizer on the dataset...
Preprocessing texts in parallel...


100%|██████████| 21092/21092 [06:00<00:00, 58.57it/s]


Saving Vectorizer and Features...
Saved successfully.


In [23]:
# (This cell is now intentionally left blank as saving is handled gracefully in the previous cell)
print("TF-IDF Vectorizer and features are handled in the previous block.")


TF-IDF Vectorizer and features are handled in the previous block.


In [24]:
x_train,x_test,y_train,y_test = train_test_split(X,finalDataset["aiGenerated"],test_size=0.40,random_state=42)

In [25]:
X.shape

(21092, 9385611)

In [26]:
x_test.shape

(8437, 9385611)

In [27]:
x_train.shape

(12655, 9385611)

In [28]:
svModel = LinearSVC(dual=True, random_state=42,C= 100,
 class_weight= 'balanced',
 loss= 'squared_hinge',
 max_iter = 5000,
 penalty= 'l2',
 tol= 0.001)
svModel.fit(x_train,y_train)


,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",True
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",100
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",5000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1

In [29]:
# Save the trained LinearSVC model using joblib
SVC_MODEL_PATH = "svModel.joblib"
joblib.dump(svModel, SVC_MODEL_PATH)
print(f"LinearSVC model successfully saved to {SVC_MODEL_PATH}")


LinearSVC model successfully saved to svModel.joblib


In [30]:
svModel.score(x_test,y_test)


0.9906364821619059

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'loss': ['hinge', 'squared_hinge'],
    'penalty': ['l2'],
    'class_weight': [None, 'balanced'],
    'tol': [1e-3, 1e-4, 1e-5],
    'max_iter': [5000, 10000]
}


In [ ]:
# grid_search = GridSearchCV(
#     estimator=LinearSVC(random_state=42),
#     param_grid=param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=2,
#     return_train_score=True
# )

In [ ]:
# grid_search.fit(x_train,y_train)

In [ ]:
# grid_search.score(x_test,y_test)

In [ ]:
# best_params = grid_search.best_params_
# best_params = {'C': 100,
#  'class_weight': 'balanced',
#  'loss': 'squared_hinge',
#  'max_iter': 5000,
#  'penalty': 'l2',
#  'tol': 0.001}

In [ ]:
test_text = """
# Confusion Matrix برای مدل LinearSVC

**Confusion Matrix** جدولی است که نشان می‌دهد مدل چگونه داده‌ها را طبقه‌بندی کرده است. این به شما کمک می‌کند مشکلات مدل را درک کنید.

## نمایش Confusion Matrix

```python
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# پیش‌بینی
y_pred = model.predict(X_test)

# محاسبه confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)
```

---

## نمایش بصری (ترجیحی)

```python
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

# رسم نمودار
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['منفی', 'مثبت'],
            yticklabels=['منفی', 'مثبت'])
plt.ylabel('واقعی')
plt.xlabel('پیش‌بینی شده')
plt.title('Confusion Matrix')
plt.show()
```

---

## تفسیر Confusion Matrix

اگر confusion matrix شما این شکل باشد:

```
[[TN  FP]
 [FN  TP]]
```

| معیار | تعریف | معنی |
|-------|--------|------|
| **TP (True Positive)** | درست مثبت | مدل صحیح گفت: مثبت |
| **TN (True Negative)** | درست منفی | مدل صحیح گفت: منفی |
| **FP (False Positive)** | غلط مثبت | مدل اشتباه گفت: مثبت (در حالی که منفی بود) |
| **FN (False Negative)** | غلط منفی | مدل اشتباه گفت: منفی (در حالی که مثبت بود) |

---

## استخراج معیارها از Confusion Matrix

```python
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

cm = confusion_matrix(y_test, y_pred)

TN = cm[0, 0]  # True Negatives
FP = cm[0, 1]  # False Positives
FN = cm[1, 0]  # False Negatives
TP = cm[1, 1]  # True Positives

# محاسبه معیارها
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)  # از مواردی که مثبت پیش‌بینی شد، چند درصد واقعاً مثبت بود
recall = TP / (TP + FN)      # از کل موارد مثبت، چند درصد را پیدا کردیم
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")
```

---

## نمونه کد کامل

```python
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# تقسیم داده‌ها
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# آموزش مدل
model = LinearSVC(random_state=42)
model.fit(X_train, y_train)

# پیش‌بینی
y_pred = model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# رسم نمودار
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel('واقعی')
plt.xlabel('پیش‌بینی شده')
plt.title('Confusion Matrix')
plt.show()

# گزارش کامل
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
```

---

## چه زمانی کدام معیار مهم است؟

| شرایط | معیار مهم |
|-------|----------|
| **کلاس‌ها متوازن هستند** | Accuracy |
| **False Positive مضر است** (مثل بیماری) | Precision |
| **False Negative مضر است** (مثل تشخیص سرطان) | Recall |
| **می‌خواهید توازن داشته باشید** | F1-Score |

آیا مسئله‌ی شما **بیماری/تشخیص** یا **دسته‌بندی دیگری** است؟
"""

In [ ]:
test_text_cleaned  = preprocess_persian_text(test_text)

In [ ]:
svModel.predict(tfidf_vectorizer.transform([test_text]))

In [ ]:
svModel.decision_function(tfidf_vectorizer.transform([test_text]))

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

y_pred = svModel.predict(x_test)

print(classification_report(y_test, y_pred))
print(f"Precision: {precision_score(y_test, y_pred)}")
print(f"Recall: {recall_score(y_test, y_pred)}")
print(f"F1-Score: {f1_score(y_test, y_pred)}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

# رسم نمودار
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['منفی', 'مثبت'],
            yticklabels=['منفی', 'مثبت'])
plt.ylabel('واقعی')
plt.xlabel('پیش‌بینی شده')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# y_pred = grid_search.predict(x_test)

# print(classification_report(y_test, y_pred))
# print(f"Precision: {precision_score(y_test, y_pred)}")
# print(f"Recall: {recall_score(y_test, y_pred)}")
# print(f"F1-Score: {f1_score(y_test, y_pred)}")


In [ ]:
# grid_search.decision_function(tfidf_vectorizer.transform([test_text]))
# #

In [ ]:
# cm = confusion_matrix(y_test, y_pred)

# # رسم نمودار
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['منفی', 'مثبت'],
#             yticklabels=['منفی', 'مثبت'])
# plt.ylabel('واقعی')
# plt.xlabel('پیش‌بینی شده')
# plt.title('Confusion Matrix')
# plt.show()

In [ ]:
clf =  LogisticRegression(random_state=42)
cross_val_score(clf,x_train,y_train,cv=10)

In [ ]:
clf.score(x_test,y_test)

### 1. Model Interpretability: Top Features (Coefficients)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get the best estimator from grid search
best_svm = grid_search.best_estimator_

# Extract coefficients and feature names
coefs = best_svm.coef_[0]
feature_names = tfidf_vectorizer.get_feature_names_out()

# Get top 20 positive (AI) and top 20 negative (Human) features
top_positive_indices = np.argsort(coefs)[-20:]
top_negative_indices = np.argsort(coefs)[:20]

top_indices = np.concatenate([top_negative_indices, top_positive_indices])
top_features = feature_names[top_indices]
top_coefs = coefs[top_indices]

# Set colors for positive vs negative
colors = ['red' if c < 0 else 'blue' for c in top_coefs]

# Plot
plt.figure(figsize=(12, 10))
plt.barh(np.arange(len(top_features)), top_coefs, color=colors)
plt.yticks(np.arange(len(top_features)), top_features)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature (3-grams)')
plt.title('Top 20 Features for Human (Red) vs AI (Blue) Text Classification')
plt.tight_layout()
plt.show()

### 2. Evaluation: ROC and Precision-Recall Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

# Use decision function for more granular curve plotting
y_scores = grid_search.decision_function(x_test)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. ROC Curve
RocCurveDisplay.from_predictions(y_test, y_scores, ax=ax[0], name='LinearSVC (GridSearch)')
ax[0].plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax[0].set_title('Receiver Operating Characteristic (ROC) Curve')
ax[0].legend()

# 2. Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(y_test, y_scores, ax=ax[1], name='LinearSVC (GridSearch)')
ax[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

### Fix Persian Text in Matplotlib
Installing required libraries and a Persian font.

In [ ]:
!pip install arabic-reshaper python-bidi
!wget -q https://github.com/rastikerdar/vazirmatn/releases/download/v33.0.0/vazirmatn-v33.0.0.zip
!unzip -q -o ./vazirmatn-v33.0.0.zip

In [ ]:
import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Define the font properties
persian_font = fm.FontProperties()

# Helper function to correct Persian text direction and shaping
def make_farsi_text(text):
    reshaped_text = arabic_reshaper.reshape(str(text))
    bidi_text = get_display(reshaped_text)
    return bidi_text


In [ ]:
# 1. Redraw Feature Importance Plot with Persian Text Fixed

# Fix text in top_features
fixed_top_features = [make_farsi_text(f) for f in top_features]

plt.figure(figsize=(12, 10))
plt.barh(np.arange(len(fixed_top_features)), top_coefs, color=colors)
# Apply the custom font to the y-ticks
plt.yticks(np.arange(len(fixed_top_features)), fixed_top_features, fontproperties=persian_font, fontsize=12)

plt.xlabel('Coefficient Value')
plt.ylabel('Feature (3-grams)')
plt.title('Top 20 Features for Human (Red) vs AI (Blue) Text Classification')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Redraw Confusion Matrix with Persian Text Fixed
plt.figure(figsize=(8, 6))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                 xticklabels=[make_farsi_text('منفی'), make_farsi_text('مثبت')],
                 yticklabels=[make_farsi_text('منفی'), make_farsi_text('مثبت')])

# Apply font to labels
plt.ylabel(make_farsi_text('واقعی'), fontproperties=persian_font, fontsize=14)
plt.xlabel(make_farsi_text('پیش‌بینی شده'), fontproperties=persian_font, fontsize=14)
plt.title('Confusion Matrix')

# Apply font to tick labels
plt.xticks(fontproperties=persian_font, fontsize=12)
plt.yticks(fontproperties=persian_font, fontsize=12)
plt.show()